# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 04: Final Evaluation & Visualisation

**Author:** Sepehr Mansouri (Experiment 2 + Evaluation Lead)
**Purpose:** Evaluate Experiment 2's best checkpoint on the 200-image
held-out final test set, generate detection visualisations, and produce
cross-experiment comparison figures for the final report.

---

### What This Notebook Does

1. **Hold-out evaluation** — Run Exp 2 best model on 100 ACD1K + 50 COD10K + 50 noise images
2. **Success criteria check** — mIoU ≥ 0.65, F1 ≥ 0.75 on ACD1K subset
3. **FPR on noise images** — False positive rate on 50 distractor images
4. **Detection visualisations** — Side-by-side: Input | Ground Truth | Prediction | Overlay
5. **Training curves** — Loss and mIoU across epochs for Stage 1 and Stage 2
6. **Cross-experiment comparison** — Bar charts comparing Exp 1 vs 2 vs 3 (if results available)

---

### Evaluation Metrics (from Proposal §2.8)

| Metric | Description |
|---|---|
| **mIoU** | Mean Intersection-over-Union (foreground + background) |
| **F1 / Dice** | Harmonic mean of precision and recall on foreground |
| **MAE** | Mean Absolute Error between prediction probability and GT mask |
| **FPR** | False Positive Rate on 50 noise distractor images |

---
## 1. Setup

In [ ]:
# Cell 1 — Ensure we're in the repo root
!git clone https://github.com/bing-er/AI-final-project
%cd /content/AI-final-project

!pip install -q transformers albumentations matplotlib

In [ ]:
# Clone Dataset

# Download ACD1K (~350 MB)
!kaggle datasets download \
    -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
    -p data/ --unzip

# Download COD10K (~1.2 GB)
!kaggle datasets download \
    -d ivanomelchenkoim11/cod10k-dataset \
    -p data/ --unzip


# Download CAMO from GitHub releases
!wget -q --show-progress \
    https://github.com/bing-er/AI-final-project/releases/download/v1.0/CAMO-V.1.0-CVIU2019.zip \
    -O data/CAMO-V.1.0-CVIU2019.zip

# Force overwrite, skip Mac junk files
!unzip -q -o data/CAMO-V.1.0-CVIU2019.zip \
    -d data/ \
    -x "*/__MACOSX/*" \
    -x "*/.DS_Store" \
    -x "*/._.DS_Store"

!rm data/CAMO-V.1.0-CVIU2019.zip
!rm -rf data/__MACOSX  # remove if it snuck in

# Verify
import os
for p in ['data/CAMO-V.1.0-CVIU2019/Images/Train',
          'data/CAMO-V.1.0-CVIU2019/Images/Test',
          'data/CAMO-V.1.0-CVIU2019/GT']:
    print('✅' if os.path.exists(p) else '❌', p)

---
## 2. Load Model & Hold-Out Data

In [ ]:
import sys, json, os
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import SegformerForSemanticSegmentation

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.dataset import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def load_model(checkpoint_path):
    """Load a SegFormer checkpoint and return the model."""
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/segformer-b2-finetuned-ade-512-512',
        num_labels=1,
        ignore_mismatched_sizes=True,
    )
    state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()

    epoch_str = str(ckpt.get('epoch', '?'))
    val_miou = ckpt.get('val_mIoU', None)
    print(f"Loaded: {checkpoint_path}")
    print(f"  Epoch: {epoch_str}  |  val mIoU: {val_miou:.4f}" if val_miou else f"  Epoch: {epoch_str}")
    return model

# Define experiments (# TODO: Yansong must add his bit)
experiments = [
    {'name': 'exp2', 'label': 'Exp 2\n(Transfer)', 'checkpoint': 'outputs/exp2/final/best_model.pth'},
    {'name': 'exp3', 'label': 'Exp 3\n(Joint)',     'checkpoint': 'outputs/exp3/final/best_model.pth'},
]

for exp in experiments:
    exp['model'] = load_model(exp['checkpoint'])

# Build hold-out datasets
print('\n[Hold-out datasets]')
subsets = {}
for name in ['acd1k', 'cod10k', 'noise']:
    try:
        subsets[name] = build_holdout_dataset('data/', name, splits_dir='splits/')
    except FileNotFoundError as e:
        print(f'  WARNING: {e}')

print(f"\nLoaded: {', '.join(f'{k}={len(v)}' for k,v in subsets.items())}")

---
## 3. Run Evaluation (Per-Image Metrics)

In [ ]:
def compute_metrics_per_image(pred_prob, mask):
    """Compute mIoU, F1, MAE for a single image."""
    pred_bin = (pred_prob > 0.5).float()

    inter    = (pred_bin * mask).sum().item()
    union    = pred_bin.sum().item() + mask.sum().item() - inter
    iou_fg   = (inter + 1e-6) / (union + 1e-6)

    inter_bg = ((1 - pred_bin) * (1 - mask)).sum().item()
    union_bg = (1 - pred_bin).sum().item() + (1 - mask).sum().item() - inter_bg
    iou_bg   = (inter_bg + 1e-6) / (union_bg + 1e-6)

    miou = (iou_fg + iou_bg) / 2
    f1   = (2 * inter + 1e-6) / (pred_bin.sum().item() + mask.sum().item() + 1e-6)
    mae  = (pred_prob - mask).abs().mean().item()

    return {'mIoU': miou, 'F1': f1, 'MAE': mae}


def evaluate_subset(model, dataset, device):
    """Run inference on a dataset, return per-image metrics + predictions."""
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        num_workers=2, pin_memory=torch.cuda.is_available())
    results = []

    with torch.no_grad():
        for batch in loader:
            image = batch['image'].to(device)
            mask  = batch['mask'].to(device)

            logits = model(pixel_values=image).logits
            upsampled = F.interpolate(logits, size=(512, 512),
                                      mode='bilinear', align_corners=False)
            prob = torch.sigmoid(upsampled)

            m = compute_metrics_per_image(prob[0].cpu(), mask[0].cpu())
            m['filename'] = batch['filename'][0]
            m['dataset']  = batch['dataset'][0]
            m['pred_prob'] = prob[0].cpu()
            m['mask']      = mask[0].cpu()
            m['image']     = image[0].cpu()
            results.append(m)

    return results


# Evaluate all experiments
for exp in experiments:
    exp['results'] = {}
    for name, ds in subsets.items():
        print(f"Evaluating {exp['name']} on {name.upper()}...")
        exp['results'][name] = evaluate_subset(exp['model'], ds, device)
        mious = [r['mIoU'] for r in exp['results'][name]]
        print(f"  {len(mious)} images  |  mean mIoU={np.mean(mious):.4f}")

# Keep backward compatibility for the rest of the notebook
all_results = experiments[0]['results']

---
## 4. Summary Table & Success Criteria

In [ ]:
def summarise(results, label):
    mious = [r['mIoU'] for r in results]
    f1s   = [r['F1']   for r in results]
    maes  = [r['MAE']  for r in results]
    return {
        'label': label, 'n': len(results),
        'mIoU_mean': np.mean(mious), 'mIoU_std': np.std(mious),
        'F1_mean':   np.mean(f1s),   'F1_std':   np.std(f1s),
        'MAE_mean':  np.mean(maes),  'MAE_std':  np.std(maes),
    }

summary = []
for name in ['acd1k', 'cod10k', 'noise']:
    if name in all_results:
        summary.append(summarise(all_results[name], name.upper()))

# Camouflage-only aggregate
camo_results = all_results.get('acd1k', []) + all_results.get('cod10k', [])
if camo_results:
    summary.append(summarise(camo_results, 'ALL CAMOUFLAGE'))

# Full 200-image aggregate
all_flat = sum(all_results.values(), [])
summary.append(summarise(all_flat, 'FULL HOLD-OUT'))

# Print table
print("=" * 60)
print("EXPERIMENT 2 — FINAL HOLD-OUT RESULTS")
print("=" * 60)
print(f"{'Subset':<25} {'N':>4} {'mIoU':>8} {'F1':>8} {'MAE':>8}")
print("-" * 60)
for r in summary:
    print(f"{r['label']:<25} {r['n']:>4} "
          f"{r['mIoU_mean']:>8.4f} {r['F1_mean']:>8.4f} {r['MAE_mean']:>8.4f}")
print("=" * 60)

# Success criteria
acd1k_row = next((r for r in summary if r['label'] == 'ACD1K'), None)
if acd1k_row:
    print("\n[Success Criteria — ACD1K hold-out]")
    miou_pass = acd1k_row['mIoU_mean'] >= 0.65
    f1_pass   = acd1k_row['F1_mean']   >= 0.75
    print(f"  mIoU >= 0.65 : {acd1k_row['mIoU_mean']:.4f}  {'✅ PASS' if miou_pass else '❌ FAIL'}")
    print(f"  F1   >= 0.75 : {acd1k_row['F1_mean']:.4f}  {'✅ PASS' if f1_pass else '❌ FAIL'}")

---
## 5. False Positive Rate (FPR) on Noise Images

The 50 noise images are ordinary outdoor scenes with no camouflage targets
(all-zero GT masks). FPR measures how many of these the model incorrectly
flags as containing a camouflaged object.

An image is considered a **false positive** if the predicted mask contains
more than 1% foreground pixels after thresholding at 0.5.

In [ ]:
if 'noise' in all_results:
    fp_threshold = 0.01  # >1% foreground pixels = false positive
    noise_results = all_results['noise']

    fp_count = 0
    fp_images = []
    for r in noise_results:
        pred_bin = (r['pred_prob'] > 0.5).float()
        fg_ratio = pred_bin.mean().item()
        if fg_ratio > fp_threshold:
            fp_count += 1
            fp_images.append((r['filename'], fg_ratio))

    fpr = fp_count / len(noise_results)
    print(f"Noise images: {len(noise_results)}")
    print(f"False positives (>{fp_threshold*100:.0f}% FG): {fp_count}")
    print(f"FPR: {fpr:.4f} ({fpr*100:.1f}%)")

    if fp_images:
        print(f"\nFalse positive filenames:")
        for fname, ratio in fp_images[:10]:
            print(f"  {fname}  ({ratio*100:.1f}% foreground)")
else:
    print("Noise subset not available — skipping FPR analysis.")

---
## 6. Detection Visualisations

Side-by-side comparison: **Input Image | Ground Truth Mask | Predicted Mask | Overlay**

The overlay shows the prediction in green (true positive), red (false positive),
and blue (false negative) on top of the original image.

In [ ]:
import matplotlib.pyplot as plt

def denormalize(img_tensor, dataset_name='ACD1K'):
    """Reverse normalization for display."""
    stats = DATASET_STATS.get(dataset_name, DATASET_STATS['ACD1K'])
    mean = torch.tensor(stats['mean']).view(3, 1, 1)
    std  = torch.tensor(stats['std']).view(3, 1, 1)
    img = img_tensor * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


def visualize_prediction(result, ax_row, dataset_name='ACD1K'):
    """Plot one row: input | GT | prediction | overlay."""
    img  = denormalize(result['image'], dataset_name)
    mask = result['mask'][0].numpy()
    pred = (result['pred_prob'][0] > 0.5).float().numpy()

    # Input
    ax_row[0].imshow(img)
    ax_row[0].set_title('Input', fontsize=9)
    ax_row[0].axis('off')

    # Ground truth
    ax_row[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    ax_row[1].set_title('Ground Truth', fontsize=9)
    ax_row[1].axis('off')

    # Prediction
    ax_row[2].imshow(pred, cmap='gray', vmin=0, vmax=1)
    ax_row[2].set_title(f"Pred (mIoU={result['mIoU']:.3f})", fontsize=9)
    ax_row[2].axis('off')

    # Overlay: TP=green, FP=red, FN=blue
    overlay = img.copy()
    tp = (pred == 1) & (mask == 1)
    fp = (pred == 1) & (mask == 0)
    fn = (pred == 0) & (mask == 1)
    alpha = 0.4
    overlay[tp] = overlay[tp] * (1 - alpha) + np.array([0, 1, 0]) * alpha  # green
    overlay[fp] = overlay[fp] * (1 - alpha) + np.array([1, 0, 0]) * alpha  # red
    overlay[fn] = overlay[fn] * (1 - alpha) + np.array([0, 0, 1]) * alpha  # blue
    ax_row[3].imshow(overlay.clip(0, 1))
    ax_row[3].set_title('Overlay (G=TP R=FP B=FN)', fontsize=9)
    ax_row[3].axis('off')

### 6a. Best ACD1K Detections (Military Camouflage)

In [ ]:
if 'acd1k' in all_results:
    # Sort by mIoU — show top 5 best
    sorted_acd = sorted(all_results['acd1k'], key=lambda x: x['mIoU'], reverse=True)

    fig, axes = plt.subplots(5, 4, figsize=(16, 20))
    fig.suptitle('Top 5 Best ACD1K Detections (Exp 2)', fontsize=14, y=0.98)
    for i, r in enumerate(sorted_acd[:5]):
        visualize_prediction(r, axes[i], 'ACD1K')
        axes[i][0].set_ylabel(r['filename'][:20], fontsize=8, rotation=0, labelpad=80)
    plt.tight_layout()
    plt.savefig('outputs/exp2/best_acd1k_detections.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/best_acd1k_detections.png")

### 6b. Worst ACD1K Detections (Failure Cases)

In [ ]:
if 'acd1k' in all_results:
    # Show bottom 5 worst
    fig, axes = plt.subplots(5, 4, figsize=(16, 20))
    fig.suptitle('Top 5 Worst ACD1K Detections — Failure Cases (Exp 2)', fontsize=14, y=0.98)
    for i, r in enumerate(sorted_acd[-5:]):
        visualize_prediction(r, axes[i], 'ACD1K')
        axes[i][0].set_ylabel(r['filename'][:20], fontsize=8, rotation=0, labelpad=80)
    plt.tight_layout()
    plt.savefig('outputs/exp2/worst_acd1k_detections.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/worst_acd1k_detections.png")

### 6c. COD10K Detections (Animal Camouflage Transfer)

In [ ]:
if 'cod10k' in all_results:
    sorted_cod = sorted(all_results['cod10k'], key=lambda x: x['mIoU'], reverse=True)
    n_show = min(5, len(sorted_cod))

    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
    fig.suptitle('COD10K Detections — Animal Camouflage (Exp 2)', fontsize=14, y=0.98)
    if n_show == 1:
        axes = [axes]
    for i, r in enumerate(sorted_cod[:n_show]):
        visualize_prediction(r, axes[i], 'COD10K')
        axes[i][0].set_ylabel(r['filename'][:20], fontsize=8, rotation=0, labelpad=80)
    plt.tight_layout()
    plt.savefig('outputs/exp2/cod10k_detections.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/cod10k_detections.png")

### 6d. Noise False Positive Examples

In [ ]:
if 'noise' in all_results:
    # Show a few noise images — both correct rejections and false positives
    noise_sorted = sorted(all_results['noise'],
                          key=lambda x: (x['pred_prob'] > 0.5).float().mean().item(),
                          reverse=True)
    n_show = min(4, len(noise_sorted))

    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
    fig.suptitle('Noise Images — False Positive Analysis (Exp 2)', fontsize=14, y=0.98)
    if n_show == 1:
        axes = [axes]
    for i, r in enumerate(noise_sorted[:n_show]):
        visualize_prediction(r, axes[i], 'COD10K')
        fg_pct = (r['pred_prob'] > 0.5).float().mean().item() * 100
        axes[i][0].set_ylabel(f"{r['filename'][:15]}\n{fg_pct:.1f}% FG",
                              fontsize=8, rotation=0, labelpad=80)
    plt.tight_layout()
    plt.savefig('outputs/exp2/noise_fp_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/noise_fp_analysis.png")

---
## 7. Training Curves

Plot loss and mIoU across epochs for both Stage 1 (COD10K pretrain) and
Stage 2 (ACD1K fine-tune).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Experiment 2 — Training Curves', fontsize=14)

stage_files = {
    'Stage 1 (COD10K)': 'outputs/exp2/stage1/history.json',
    'Stage 2 (ACD1K)':  'outputs/exp2/final/history.json',
}

for col, (stage_name, path) in enumerate(stage_files.items()):
    if not os.path.exists(path):
        print(f"  {path} not found, skipping {stage_name}")
        continue

    with open(path) as f:
        h = json.load(f)

    epochs     = [r['epoch'] for r in h]
    train_loss = [r['train_loss'] for r in h]
    val_loss   = [r['val_loss'] for r in h]
    train_miou = [r['train_mIoU'] for r in h]
    val_miou   = [r['val_mIoU'] for r in h]

    # Loss
    axes[0][col].plot(epochs, train_loss, label='Train', linewidth=1.5)
    axes[0][col].plot(epochs, val_loss, label='Val', linewidth=1.5)
    axes[0][col].set_title(f'{stage_name} — Loss')
    axes[0][col].set_xlabel('Epoch')
    axes[0][col].set_ylabel('Loss')
    axes[0][col].legend()
    axes[0][col].grid(True, alpha=0.3)

    # mIoU
    axes[1][col].plot(epochs, train_miou, label='Train', linewidth=1.5)
    axes[1][col].plot(epochs, val_miou, label='Val', linewidth=1.5)
    best = max(h, key=lambda x: x['val_mIoU'])
    axes[1][col].axhline(y=best['val_mIoU'], color='r', linestyle='--', alpha=0.5,
                         label=f"Best={best['val_mIoU']:.4f} @ep{best['epoch']}")
    axes[1][col].set_title(f'{stage_name} — mIoU')
    axes[1][col].set_xlabel('Epoch')
    axes[1][col].set_ylabel('mIoU')
    axes[1][col].legend()
    axes[1][col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/exp2/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → outputs/exp2/training_curves.png")

---
## 8. Cross-Experiment Comparison

Compare Exp 1 (SINetV2), Exp 2 (Transfer Learning), and Exp 3 (Joint Training)
on the ACD1K hold-out subset. This cell only runs if eval results from the
other experiments are available.

In [ ]:
import matplotlib.pyplot as plt

# Build comparison from in-memory experiment results
exp_results = {}
for exp in experiments:
    acd1k = exp['results'].get('acd1k', [])
    if acd1k:
        s = summarise(acd1k, 'ACD1K')
        exp_results[exp['label']] = {
            'mIoU': s['mIoU_mean'],
            'F1':   s['F1_mean'],
            'MAE':  s['MAE_mean'],
        }

if len(exp_results) >= 2:
    labels = list(exp_results.keys())
    metrics = ['mIoU', 'F1', 'MAE']
    colors = ['#2196F3', '#4CAF50', '#FF9800']

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('Cross-Experiment Comparison — ACD1K Hold-Out', fontsize=14)

    for i, metric in enumerate(metrics):
        values = [exp_results[l][metric] for l in labels]
        bars = axes[i].bar(labels, values, color=colors[:len(labels)], alpha=0.8, width=0.5)
        axes[i].set_title(metric, fontsize=12)
        axes[i].set_ylim(0, 1.0 if metric != 'MAE' else max(values) * 1.5)

        for bar, val in zip(bars, values):
            axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.4f}', ha='center', va='bottom', fontsize=10)

        if metric == 'mIoU':
            axes[i].axhline(y=0.65, color='red', linestyle='--', alpha=0.6, label='Target ≥ 0.65')
            axes[i].legend(fontsize=9)
        elif metric == 'F1':
            axes[i].axhline(y=0.75, color='red', linestyle='--', alpha=0.6, label='Target ≥ 0.75')
            axes[i].legend(fontsize=9)

        axes[i].grid(True, alpha=0.2, axis='y')

    plt.tight_layout()
    plt.savefig('outputs/exp2/cross_experiment_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/cross_experiment_comparison.png")
else:
    print(f"Only {len(exp_results)} experiment(s) available — need ≥ 2 for comparison.")
    print("Run evaluate.py for the other experiments before proceeding.")

---
## 8b. Side-by-Side Image Comparison Across Experiments

Shows the **same ACD1K image** predicted by each experiment side-by-side.
This directly reveals where one model succeeds and another fails.

Columns: **Original | Ground Truth | Exp 2 Overlay | Exp 3 Overlay**

Overlay colours: Green = correct detection (TP), Red = false alarm (FP), Blue = missed target (FN)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def side_by_side_comparison(experiments, subsets, dataset_name='acd1k',
                            stats_key='ACD1K', num_images=6, seed=42):
    """Show same images across all experiments for direct comparison."""
    ds = subsets[dataset_name]
    stats = DATASET_STATS[stats_key]
    n_exps = len(experiments)

    np.random.seed(seed)
    indices = np.random.choice(len(ds), min(num_images, len(ds)), replace=False)

    fig, axes = plt.subplots(len(indices), n_exps + 2,
                             figsize=(4*(n_exps+2), 4*len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Cross-Experiment Comparison — Same {stats_key} Images',
                 fontsize=16, y=1.01)

    # Column headers
    axes[0][0].set_title('Original', fontsize=11, fontweight='bold')
    axes[0][1].set_title('Ground Truth', fontsize=11, fontweight='bold')
    for col, exp in enumerate(experiments):
        short_label = exp['name'].upper()
        axes[0][col+2].set_title(short_label, fontsize=11, fontweight='bold')

    for row, idx in enumerate(indices):
        batch = ds[idx]
        img_tensor = batch['image']
        mask_np = batch['mask'][0].numpy()
        fname = batch['filename']

        # Denormalize for display
        mean = torch.tensor(stats['mean']).view(3,1,1)
        std  = torch.tensor(stats['std']).view(3,1,1)
        img_display = (img_tensor * std + mean).clamp(0,1).permute(1,2,0).numpy()

        # Col 0: Original
        axes[row][0].imshow(img_display)
        axes[row][0].set_ylabel(fname[:20], fontsize=8, rotation=0, labelpad=80)
        axes[row][0].axis('off')

        # Col 1: Ground truth
        axes[row][1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
        axes[row][1].axis('off')

        # Col 2+: Each experiment's prediction overlay
        for col, exp in enumerate(experiments):
            with torch.no_grad():
                logits = exp['model'](pixel_values=img_tensor.unsqueeze(0).to(device)).logits
                upsampled = F.interpolate(logits, size=(512,512),
                                         mode='bilinear', align_corners=False)
                prob = torch.sigmoid(upsampled).squeeze().cpu().numpy()

            pred_bin = (prob > 0.5).astype(np.float32)

            # Compute mIoU for this image
            inter = (pred_bin * mask_np).sum()
            union = pred_bin.sum() + mask_np.sum() - inter
            iou_fg = (inter + 1e-6) / (union + 1e-6)
            inter_bg = ((1-pred_bin) * (1-mask_np)).sum()
            union_bg = (1-pred_bin).sum() + (1-mask_np).sum() - inter_bg
            iou_bg = (inter_bg + 1e-6) / (union_bg + 1e-6)
            miou = (iou_fg + iou_bg) / 2

            # Build overlay
            overlay = img_display.copy()
            tp = (pred_bin > 0.5) & (mask_np > 0.5)
            fp = (pred_bin > 0.5) & (mask_np < 0.5)
            fn = (pred_bin < 0.5) & (mask_np > 0.5)
            alpha = 0.4
            overlay[tp] = overlay[tp] * (1-alpha) + np.array([0,1,0]) * alpha
            overlay[fp] = overlay[fp] * (1-alpha) + np.array([1,0,0]) * alpha
            overlay[fn] = overlay[fn] * (1-alpha) + np.array([0,0,1]) * alpha

            axes[row][col+2].imshow(overlay.clip(0,1))
            axes[row][col+2].set_title(f'mIoU={miou:.3f}', fontsize=9)
            axes[row][col+2].axis('off')

    # Legend
    legend_elements = [Patch(facecolor='green', label='True Positive'),
                       Patch(facecolor='red',   label='False Positive'),
                       Patch(facecolor='blue',  label='Missed (FN)')]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3,
               fontsize=11, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(f'outputs/exp2/side_by_side_{dataset_name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → outputs/exp2/side_by_side_{dataset_name}.png')

print('✅ Side-by-side comparison function ready')

In [ ]:
# ACD1K side-by-side (military camouflage)
print('ACD1K — Same images across experiments:')
side_by_side_comparison(experiments, subsets, 'acd1k', 'ACD1K', num_images=6)

In [ ]:
#  COD10K side-by-side (animal camouflage transfer)
print('COD10K — Same images across experiments:')
side_by_side_comparison(experiments, subsets, 'cod10k', 'COD10K', num_images=4)

---
## 9. Per-Image mIoU Distribution

In [ ]:
if 'acd1k' in all_results:
    import matplotlib.pyplot as plt

    mious = [r['mIoU'] for r in all_results['acd1k']]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(mious, bins=30, color='#2196F3', alpha=0.7, edgecolor='white')
    ax.axvline(np.mean(mious), color='red', linestyle='--',
               label=f'Mean = {np.mean(mious):.4f}')
    ax.axvline(0.65, color='orange', linestyle='--', alpha=0.6,
               label='Target ≥ 0.65')
    ax.set_xlabel('mIoU')
    ax.set_ylabel('Count')
    ax.set_title('Exp 2 — Per-Image mIoU Distribution (ACD1K Hold-Out)')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig('outputs/exp2/miou_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → outputs/exp2/miou_distribution.png")

---
## 10. Save Results to JSON

In [ ]:
# Strip tensors before saving (not JSON-serializable)
for exp in experiments:
    save_results = {}
    for subset_name, results in exp['results'].items():
        save_results[subset_name] = [
            {k: v for k, v in r.items() if k not in ('pred_prob', 'mask', 'image')}
            for r in results
        ]

    exp_summary = []
    for name in ['acd1k', 'cod10k', 'noise']:
        if name in exp['results']:
            exp_summary.append(summarise(exp['results'][name], name.upper()))

    out = {
        'experiment': exp['name'],
        'checkpoint': exp['checkpoint'],
        'summary':    exp_summary,
        'per_image':  save_results,
    }

    out_dir = f"outputs/{exp['name']}/eval"
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/eval_results.json"
    with open(out_path, 'w') as f:
        json.dump(out, f, indent=2, default=lambda x: round(float(x), 6))
    print(f"Saved → {out_path}")